# Results v2: every block, every dataset

CPU runtime is enough. **Runtime → Run all.** Takes ~40–60 min (mostly unpacking ~8 GB).

At the end everything is zipped to Drive `sparse3dgs/results_v2_tables.zip`. Paste Claude the printed **verdicts** (the cells marked 📋).

## 0. Drive and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cd /content && rm -rf dngs && git clone -q -b claude/project-thread-gbqx9f https://github.com/aayxsh23/dngs-dropout.git dngs && cd dngs && git log -1 --format='code @ %h %s'
!pip -q install statsmodels
%cd /content/dngs
!python experiments/bias_variance.py --self_test && python experiments/interaction_scales.py --self_test && python experiments/extra_blocks.py --self_test | tail -1

## 1. Find, check and unpack the tarballs
Each file must be in your Drive (if one is 'MISSING', open the sparse3dgs folder → right-click it → Organize → Add shortcut → My Drive, then rerun this cell). Every md5 line must say OK.

In [ ]:
import glob, hashlib, os, subprocess
WANT = {  # name: md5 (prefix is enough)
    'results_full_nockpt.tgz': '2ea56c51',          # T1, pass, dose, noise (fern)
    'views_s12_nockpt.tgz':    'f13a3e120e7dbfae80b21c25ca3e7ba2',
    'results_v11.tgz':         'dc711089c49b385b2d20a8ad511c0142',  # comp, depthdose, noise2
    'eval_dtu.tgz':            '64aa583949ea46b47379fe82adfdd1ed',
    'eval_blender.tgz':        '21f7f25478a865419aee65c65b9c18a2',
}
os.makedirs('/content/res', exist_ok=True)
found = {}
for name, md5 in WANT.items():
    hits = sorted(glob.glob('/content/drive/MyDrive/**/' + name, recursive=True))
    if not hits:
        print('MISSING', name); continue
    found[name] = hits[0]
    print('found', name, '->', hits[0])
for name, path in found.items():
    local = '/content/' + name
    subprocess.run(['cp', path, local], check=True)
    h = hashlib.md5()
    with open(local, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    ok = h.hexdigest().startswith(WANT[name])
    print(('OK  ' if ok else 'BAD ') + h.hexdigest(), name)
    subprocess.run(['tar', 'xzf', local, '-C', '/content/res'], check=True)
    os.remove(local)
!du -sh /content/res/* 2>/dev/null

## 2. Locate each block
Prints the folder found for each block and how many finished runs (metrics.json) it holds. Expected: T1 120, comp 96, depthdose 96, noise2 48, dtu 225, blender 120, views 360.

In [ ]:
import re
def runs_under(root):
    return glob.glob(root + '/*_*v/*/*/seed*/metrics.json')
cands = sorted(set(os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/res/**/*_*v/', recursive=True)))
roots = {}
for r in cands:
    base = os.path.basename(r)
    sub = os.listdir(r)
    key = {'outputs': 'T1', 'outputs_views': 'views', 'outputs_comp': 'comp', 'outputs_depthdose': 'depthdose',
           'outputs_noise2': 'noise2', 'outputs_noise': 'noise', 'outputs_dtu': 'dtu', 'outputs_blender': 'blender',
           'outputs_pass': 'pass', 'outputs_dose': 'dose'}.get(base)
    if key is None and any(s.startswith('dtu_') for s in sub): key = 'dtu'
    if key is None and any(s.startswith('blender_') for s in sub): key = 'blender'
    if key and key not in roots:
        roots[key] = r
for k, r in roots.items():
    print('{:<10} {:>4} runs  {}'.format(k, len(runs_under(r)), r))

## 3. Aggregate every block

In [ ]:
T = '/content/tables'
for k, r in roots.items():
    extra = ['--seeds', '0'] if k in ('noise', 'noise2') else []
    !python experiments/aggregate.py --out_root {r} --tables {T}/{k} {' '.join(extra)} 2>&1 | tail -2
!wc -l {T}/*/runs.csv

## 4. 📋 Factorial per dataset + pooled (X4, X5, H-PD)

In [ ]:
!python experiments/stats.py --runs {T}/T1/runs.csv --tables {T}/llff --noise_runs {T}/noise/runs.csv 2>&1 | tail -3
for ds, nv in (('dtu', 3), ('blender', 8)):
    if ds in roots:
        !python experiments/stats.py --dataset {ds} --n_views {nv} --runs {T}/{ds}/runs.csv --tables {T}/{ds} 2>&1 | tail -3
pooled = 'llff={}/T1/runs.csv'.format(T) + ''.join(' {0}={1}/{0}/runs.csv'.format(d, T) for d in ('dtu', 'blender') if d in roots)
!python experiments/stats.py --pooled {pooled} --blender_views 8 --tables {T}/pooled 2>&1 | tail -3
for f in ('{}/llff/verdict_3v.txt', '{}/dtu/verdict_dtu_3v.txt', '{}/blender/verdict_blender_8v.txt', '{}/pooled/verdict_pooled.txt'):
    f = f.format(T)
    if os.path.exists(f):
        print('===', f); print(open(f).read())

## 5. 📋 Compensation, depth dose, noise floor, power (extra_blocks.py)

In [ ]:
args = '--runs {}/T1/runs.csv'.format(T)
for k, opt in (('comp', 'comp_runs'), ('depthdose', 'ddose_runs'), ('noise2', 'noise2_runs'), ('noise', 'noise_runs')):
    if k in roots:
        args += ' --{} {}/{}/runs.csv'.format(opt, T, k)
!python experiments/extra_blocks.py {args} --tables {T}/extra

## 6. 📋 Error budget N1 (BV1–BV6) and N4 on every dataset
The slowest step (reads every render).

In [ ]:
!git pull -q && git log -1 --format='code @ %h %s'
EB = T + '/error_budget'
jobs = [('T1', 'llff', 3), ('dtu', 'dtu', 3), ('blender', 'blender', 8)] + [('views', 'llff', n) for n in (2, 6, 9)]
for k, ds, nv in jobs:
    if k not in roots: continue
    !python experiments/bias_variance.py --out_root {roots[k]} --dataset {ds} --n_views {nv} --tables {EB} 2>&1 | tail -2
for k, ds, nv in (('T1', 'llff', 3), ('dtu', 'dtu', 3), ('blender', 'blender', 8)):
    br = '{}/error_budget_runs_{}_{}v.csv'.format(EB, ds, nv)
    if k not in roots or not os.path.exists(br):
        print('N4 skipped for', ds, '(no error budget)'); continue
    !python experiments/interaction_scales.py --runs {T}/{k}/runs.csv --budget_runs {br} --dataset {ds} --n_views {nv} --tables {T}/n4
for f in sorted(glob.glob(EB + '/error_budget_verdict_*.txt')) + sorted(glob.glob(T + '/n4/*verdict*.txt')):
    print('===', f); print(open(f).read())

## 7a. Figures (F1, F2, F3, F5, F7, F8, F9)

In [ ]:
!python experiments/plots/v2_figures.py --tables {T} --figures {T}/figures
from IPython.display import Image, display
for f in sorted(glob.glob(T + '/figures/*.png')):
    print(f); display(Image(f, width=700))

## 7b. Save to Drive

In [ ]:
!cd /content && rm -f results_v2_tables.zip && zip -qr results_v2_tables.zip tables && cp results_v2_tables.zip /content/drive/MyDrive/sparse3dgs/ && ls -l /content/drive/MyDrive/sparse3dgs/results_v2_tables.zip && echo SAVED